# TrackNetV3 — Fine-tuning con PadelTracker100

**Run All.** Cada celda detecta si ya tiene su output y se salta el trabajo.

| Celda | Qué hace | Skip si... |
|-------|----------|------------|
| A | GPU + Drive + variables | — (siempre corre, es setup) |
| B | Descarga PadelTracker100 (~7 GB) | mp4s ya en Drive |
| C | Parcha TrackNetV3 + pesos | — (siempre corre, parches son idempotentes) |
| D | Extrae frames (~40 min) | frames ya en Drive |
| E | Convierte JSON → CSV | CSVs ya en Drive |
| F | Crea splits val/ y test/ | splits ya en Drive |
| G | **Fine-tuning** (~3-4h) | — (siempre corre, retoma desde checkpoint) |
| H | Inferencia sobre tu vídeo | — (corre si hay vídeo de test en Drive, si no avisa) |
| I | Resultados | — (muestra si hay predicciones, si no avisa) |

In [ ]:
# ============================================================
# CELDA A — Setup: GPU + Drive + variables globales
# Siempre corre — solo define variables y monta Drive
# ============================================================
import subprocess, os

r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                   capture_output=True, text=True)
print('GPU:', r.stdout.strip() if r.returncode == 0 else '❌ SIN GPU — activa T4 en Runtime > Change runtime type')

from google.colab import drive
drive.mount('/content/drive')

# ── Variables globales ── usadas en todas las celdas ─────────
DRIVE_ROOT        = '/content/drive/MyDrive/paddelstats_tracknet'
DATASET_DIR       = f'{DRIVE_ROOT}/padeltracker100'
LABELS_DIR        = f'{DATASET_DIR}/labels'
TRACKNET_DATA_DIR = f'{DRIVE_ROOT}/tracknetv3_dataset'
TRACKNET_DIR      = f'{DRIVE_ROOT}/TrackNetV3'
CKPT_DIR          = f'{DRIVE_ROOT}/ckpts'
EXP_DIR           = f'{DRIVE_ROOT}/exp_padel'
PRED_DIR          = f'{DRIVE_ROOT}/predictions'
TEST_VIDEO        = f'{DRIVE_ROOT}/test_video.mp4'   # sube tu vídeo aquí
TARGET_W, TARGET_H = 960, 540
EPOCHS            = 10

MATCH_CONFIG = [
    ('match1', '2022_BCN_FinalF_1', f'{LABELS_DIR}/2022_BCN_FinalF_1_ball.json'),
    ('match2', '2022_BCN_FinalM_1', f'{LABELS_DIR}/2022_BCN_FinalM_1_ball.json'),
]

for d in [DRIVE_ROOT, DATASET_DIR, LABELS_DIR, TRACKNET_DATA_DIR, CKPT_DIR, EXP_DIR, PRED_DIR]:
    os.makedirs(d, exist_ok=True)

print('✅ Setup completo')

In [ ]:
# ============================================================
# CELDA B — Descargar PadelTracker100 desde Zenodo
# Skip automático si los mp4s ya están en Drive
# ============================================================
import os, glob, shutil

needed = [
    f'{DATASET_DIR}/2022_BCN_FinalF_1.mp4',
    f'{DATASET_DIR}/2022_BCN_FinalM_1.mp4',
    f'{LABELS_DIR}/2022_BCN_FinalF_1_ball.json',
    f'{LABELS_DIR}/2022_BCN_FinalM_1_ball.json',
]
missing = [f for f in needed if not os.path.exists(f) or os.path.getsize(f) < 1000]

if not missing:
    print('✅ Dataset ya en Drive — saltando descarga')
    for f in needed:
        print(f'   {os.path.basename(f)} ({os.path.getsize(f)/1e6:.0f} MB)')
else:
    print(f'Faltan {len(missing)} archivos, descargando...')
    ZIP_PATH = f'{DATASET_DIR}/padel-data-labels.zip'
    ZIP_URL  = 'https://zenodo.org/records/14653706/files/padel-data-labels.zip?download=1'

    if not os.path.exists(ZIP_PATH) or os.path.getsize(ZIP_PATH) < 1e8:
        print('Descargando padel-data-labels.zip (~7.1 GB)...')
        !wget -q --show-progress -O "{ZIP_PATH}" "{ZIP_URL}"

    print('Extrayendo...')
    !unzip -q "{ZIP_PATH}" -d "{DATASET_DIR}"

    for mp4 in glob.glob(f'{DATASET_DIR}/**/*.mp4', recursive=True):
        dst = os.path.join(DATASET_DIR, os.path.basename(mp4))
        if mp4 != dst: shutil.move(mp4, dst)
    for j in glob.glob(f'{DATASET_DIR}/**/*.json', recursive=True):
        dst = os.path.join(LABELS_DIR, os.path.basename(j))
        if j != dst: shutil.move(j, dst)

    still_missing = [f for f in needed if not os.path.exists(f) or os.path.getsize(f) < 1000]
    if still_missing:
        print('❌ Siguen faltando:', still_missing)
    else:
        print('✅ Dataset listo')

In [ ]:
# ============================================================
# CELDA C — Preparar TrackNetV3
# Siempre corre — parches son idempotentes, checkpoint se verifica
# ============================================================
import os, re, shutil, torch, gdown
from pathlib import Path

!pip install parse --quiet

# 1. Clonar TrackNetV3 si no existe
if not os.path.exists(f'{TRACKNET_DIR}/train.py'):
    !git clone https://github.com/qaz812345/TrackNetV3.git "{TRACKNET_DIR}" --quiet
    print('✅ TrackNetV3 clonado')
else:
    print('✅ TrackNetV3 ya existe')

# 2. Parchear dataset.py — siempre, re.sub garantiza idempotencia
dataset_py = f'{TRACKNET_DIR}/dataset.py'
with open(dataset_py) as f:
    src = f.read()
src = re.sub(r"data_dir = '.*?'", f"data_dir = '{TRACKNET_DATA_DIR}'", src)
src = src.replace(
    "w, h = Image.open(os.path.join(rally_dir, f'0.{IMG_FORMAT}')).size",
    "first = sorted([x for x in os.listdir(rally_dir) if x.endswith(IMG_FORMAT)])[0]; w, h = Image.open(os.path.join(rally_dir, first)).size"
)
src = src.replace(
    "f_file = np.array([os.path.join(rally_dir, f'{f_id}.{IMG_FORMAT}') for f_id in label_df['Frame']])",
    "f_file = np.array([os.path.join(rally_dir, f'{int(f_id):06d}.{IMG_FORMAT}') for f_id in label_df['Frame']])"
)
with open(dataset_py, 'w') as f:
    f.write(src)
print(f'✅ dataset.py parcheado → data_dir={TRACKNET_DATA_DIR}')

# 3. Parchear utils/general.py — IMG_FORMAT png → jpg
general_py = f'{TRACKNET_DIR}/utils/general.py'
with open(general_py) as f:
    src = f.read()
src = re.sub(r"IMG_FORMAT = '.*?'", "IMG_FORMAT = 'jpg'", src)
with open(general_py, 'w') as f:
    f.write(src)
print('✅ utils/general.py parcheado → IMG_FORMAT=jpg')

# 4. Descargar pesos si no existen
tracknet_pt = f'{CKPT_DIR}/TrackNet_best.pt'
if not os.path.exists(tracknet_pt):
    print('Descargando pesos pre-entrenados...')
    ckpt_zip = f'{CKPT_DIR}/ckpts.zip'
    gdown.download(id='1CfzE87a0f6LhBp0kniSl1-89zaLCZ8cA', output=ckpt_zip, quiet=False)
    !unzip -q "{ckpt_zip}" -d "{CKPT_DIR}"
    for pt in Path(CKPT_DIR).rglob('*.pt'):
        dst = f'{CKPT_DIR}/{pt.name}'
        if str(pt) != dst: shutil.move(str(pt), dst)
    print('✅ Pesos descargados')
else:
    print('✅ Pesos ya descargados')

# Copiar pesos a exp_dir si no están
for pt_name in ['TrackNet_best.pt', 'InpaintNet_best.pt']:
    src_pt, dst_pt = f'{CKPT_DIR}/{pt_name}', f'{EXP_DIR}/{pt_name}'
    if os.path.exists(src_pt) and not os.path.exists(dst_pt):
        shutil.copy(src_pt, dst_pt)

# 5. Verificar/crear TrackNet_cur.pt con las claves necesarias
cur_pt = f'{EXP_DIR}/TrackNet_cur.pt'
if not os.path.exists(cur_pt):
    shutil.copy(f'{EXP_DIR}/TrackNet_best.pt', cur_pt)
    print('✅ TrackNet_cur.pt creado desde best')

ckpt = torch.load(cur_pt, map_location='cpu')
updated = False
for k, v in {'mask_ratio': 0, 'max_val_acc': 0}.items():
    if k not in ckpt.get('param_dict', {}):
        ckpt.setdefault('param_dict', {})[k] = v; updated = True
if 'max_val_acc' not in ckpt:
    ckpt['max_val_acc'] = 0; updated = True
if updated:
    torch.save(ckpt, cur_pt)
    print('✅ Checkpoint actualizado con claves faltantes')
else:
    print('✅ Checkpoint OK')

# 6. Instalar dependencias
!pip install -r "{TRACKNET_DIR}/requirements.txt" --quiet 2>&1 | tail -2

print('\n✅ Celda C completada')

In [ ]:
# ============================================================
# CELDA D — Extraer frames de los mp4s con ffmpeg
# Skip automático si los frames ya están en Drive
# Reanuda automáticamente si se cortó a medias
# Estrategia: extrae a /tmp (SSD local) y copia a Drive de golpe
# ============================================================
import os, shutil, subprocess
from pathlib import Path

VIDEOS = [
    ('match1', '2022_BCN_FinalF_1', f'{DATASET_DIR}/2022_BCN_FinalF_1.mp4'),
    ('match2', '2022_BCN_FinalM_1', f'{DATASET_DIR}/2022_BCN_FinalM_1.mp4'),
]

def get_video_info(video_path):
    """Devuelve (total_frames, fps) usando ffprobe."""
    r = subprocess.run(
        ['ffprobe', '-v', 'error', '-select_streams', 'v:0',
         '-show_entries', 'stream=nb_frames,r_frame_rate',
         '-of', 'default=noprint_wrappers=1', video_path],
        capture_output=True, text=True)
    frames, fps = None, 25.0
    for line in r.stdout.splitlines():
        if line.startswith('nb_frames='):
            try: frames = int(line.split('=')[1])
            except: pass
        if line.startswith('r_frame_rate='):
            try:
                num, den = map(int, line.split('=')[1].split('/'))
                fps = num / den
            except: pass
    return frames, fps

for match_name, video_name, video_path in VIDEOS:
    if not os.path.exists(video_path):
        print(f'⚠️  {match_name}: mp4 no encontrado — ejecuta celda B primero')
        continue

    out_dir = f'{TRACKNET_DATA_DIR}/train/{match_name}/frame/{video_name}'
    os.makedirs(out_dir, exist_ok=True)

    existing = sorted(Path(out_dir).glob('*.jpg'))
    total, fps = get_video_info(video_path)
    last_idx = int(existing[-1].stem) if existing else -1

    if total and last_idx >= total - 10:
        print(f'✅ {match_name}: {len(existing)} frames ya extraídos')
        continue

    start_frame = last_idx + 1
    tmp_dir = f'/tmp/frames_{match_name}'
    # Limpiar /tmp de runs anteriores interrumpidas
    if os.path.exists(tmp_dir):
        shutil.rmtree(tmp_dir)
    os.makedirs(tmp_dir)

    print(f'Extrayendo {match_name} desde frame {start_frame}/{total} → /tmp...')

    if start_frame == 0:
        # Extracción completa desde el inicio
        cmd = [
            'ffmpeg', '-y', '-i', video_path,
            '-vf', f'scale={TARGET_W}:{TARGET_H}',
            '-q:v', '2', '-start_number', '0',
            f'{tmp_dir}/%06d.jpg'
        ]
    else:
        # Reanudación: seek rápido con -ss ANTES de -i (keyframe-accurate, O(1) memoria)
        # No usar select filter — decodifica todo el vídeo en RAM y explota
        start_time = start_frame / fps
        cmd = [
            'ffmpeg', '-y',
            '-ss', f'{start_time:.3f}',  # seek rápido al keyframe más cercano
            '-i', video_path,
            '-vf', f'scale={TARGET_W}:{TARGET_H}',
            '-q:v', '2', '-start_number', str(start_frame),
            f'{tmp_dir}/%06d.jpg'
        ]

    print('  ffmpeg corriendo...')
    ret = subprocess.run(cmd, stderr=subprocess.PIPE, text=True)
    if ret.returncode != 0:
        print(f'❌ ffmpeg error:\n{ret.stderr[-800:]}')
        continue

    # Copiar de /tmp a Drive de golpe, saltando los que ya existen
    tmp_frames = sorted(Path(tmp_dir).glob('*.jpg'))
    new_frames = [f for f in tmp_frames if not (Path(out_dir) / f.name).exists()]
    print(f'  Copiando {len(new_frames)} frames nuevos a Drive ({len(tmp_frames)} extraídos)...')
    for f in new_frames:
        shutil.copy2(f, Path(out_dir) / f.name)
    os.sync()
    shutil.rmtree(tmp_dir)

    total_in_drive = len(list(Path(out_dir).glob('*.jpg')))
    print(f'✅ {match_name}: {total_in_drive} frames en Drive')

print('\n✅ Celda D completada')

In [ ]:
# ============================================================
# CELDA E — Convertir anotaciones COCO JSON → CSV TrackNetV3
# Skip automático si los CSVs ya están en Drive
# ============================================================
import json, os, time, shutil
import pandas as pd
from pathlib import Path

ORIG_W, ORIG_H   = 1920, 1080
BALL_CATEGORY_ID = 1

def convert_ball_json(json_path, output_csv):
    with open(json_path) as f:
        data = json.load(f)
    img_map = {}
    for img in data.get('images', []):
        stem = Path(img['file_name']).stem
        frame_num = int(''.join(filter(str.isdigit, stem)))
        img_map[img['id']] = {'frame': frame_num,
                               'orig_w': img.get('width', ORIG_W),
                               'orig_h': img.get('height', ORIG_H)}
    ball_anns = {}
    for ann in data.get('annotations', []):
        if ann.get('category_id') != BALL_CATEGORY_ID: continue
        img_id = ann['image_id']
        bbox = ann.get('bbox')
        if bbox and img_id in img_map:
            info = img_map[img_id]
            sx, sy = TARGET_W / info['orig_w'], TARGET_H / info['orig_h']
            ball_anns[img_id] = (round((bbox[0] + bbox[2]/2) * sx),
                                  round((bbox[1] + bbox[3]/2) * sy))
    rows = []
    for img_id, info in sorted(img_map.items(), key=lambda x: x[1]['frame']):
        if img_id in ball_anns:
            cx, cy = ball_anns[img_id]
            rows.append({'Frame': info['frame'], 'Visibility': 1, 'X': cx, 'Y': cy})
        else:
            rows.append({'Frame': info['frame'], 'Visibility': 0, 'X': 0, 'Y': 0})
    df = pd.DataFrame(rows)
    os.makedirs(os.path.dirname(output_csv), exist_ok=True)
    tmp = f'/tmp/{Path(output_csv).name}'
    df.to_csv(tmp, index=False)
    shutil.copy2(tmp, output_csv)
    os.sync()
    return len(df), (df['Visibility'] == 1).sum()

for match_name, video_name, json_path in MATCH_CONFIG:
    if not os.path.exists(json_path):
        print(f'⚠️  {match_name}: JSON no encontrado — ejecuta celda B primero')
        continue

    csv_dir = f'{TRACKNET_DATA_DIR}/train/{match_name}/csv'
    out_csv = f'{csv_dir}/{video_name}_ball.csv'

    if os.path.exists(out_csv) and os.path.getsize(out_csv) > 1000:
        df = pd.read_csv(out_csv)
        n_vis = (df['Visibility'] == 1).sum()
        print(f'✅ {match_name}: CSV ya existe ({len(df)} frames, {n_vis} con pelota)')
        continue

    n_frames, n_vis = convert_ball_json(json_path, out_csv)
    print(f'✅ {match_name}: {n_frames} frames, {n_vis} con pelota ({n_vis/n_frames*100:.1f}%)')

print('\n✅ Celda E completada')

In [ ]:
# ============================================================
# CELDA F — Crear splits val/ y test/
# Skip automático si los splits ya están en Drive
# ============================================================
import os, shutil, pandas as pd
from pathlib import Path

def create_split(split, src_match, src_video, n_frames):
    dst_frame_dir = f'{TRACKNET_DATA_DIR}/{split}/match1/frame/{src_video}'
    dst_csv_dir   = f'{TRACKNET_DATA_DIR}/{split}/match1/csv'
    dst_csv       = f'{dst_csv_dir}/{src_video}_ball.csv'

    existing_frames = list(Path(dst_frame_dir).glob('*.jpg')) if os.path.exists(dst_frame_dir) else []
    if len(existing_frames) >= n_frames and os.path.exists(dst_csv):
        print(f'✅ {split}/match1 ya existe ({len(existing_frames)} frames)')
        return

    src_frame_dir = f'{TRACKNET_DATA_DIR}/train/{src_match}/frame/{src_video}'
    src_csv       = f'{TRACKNET_DATA_DIR}/train/{src_match}/csv/{src_video}_ball.csv'

    if not os.path.exists(src_frame_dir) or not os.path.exists(src_csv):
        print(f'⚠️  {split}: fuente no disponible — ejecuta celdas D y E primero')
        return

    os.makedirs(dst_frame_dir, exist_ok=True)
    os.makedirs(dst_csv_dir, exist_ok=True)

    src_frames = sorted(Path(src_frame_dir).glob('*.jpg'))[-n_frames:]
    print(f'Creando {split}/match1 ({len(src_frames)} frames)...')
    for f in src_frames:
        dst = Path(dst_frame_dir) / f.name
        if not dst.exists(): shutil.copy2(f, dst)

    existing = sorted([int(f.stem) for f in Path(dst_frame_dir).glob('*.jpg')])
    df = pd.read_csv(src_csv)
    df[df['Frame'].isin(existing)].to_csv(dst_csv, index=False)
    print(f'✅ {split}/match1 creado ({len(existing)} frames)')

create_split('val',  'match2', '2022_BCN_FinalM_1', 500)
create_split('test', 'match1', '2022_BCN_FinalF_1', 300)

print('\n✅ Celda F completada')

In [ ]:
# ============================================================
# CELDA G — Fine-tuning
# Siempre corre — --resume_training retoma desde el último checkpoint
# ============================================================
import os

# Verificar que los datos están listos
errors = []
for split in ['train', 'val', 'test']:
    if not os.path.exists(f'{TRACKNET_DATA_DIR}/{split}'):
        errors.append(f'❌ Falta split {split}/ — ejecuta celdas D, E, F')
if errors:
    for e in errors: print(e)
    raise SystemExit('Datos no listos')

os.chdir(TRACKNET_DIR)
print(f'Training — {EPOCHS} epochs | checkpoints → {EXP_DIR}')
print('Retoma automáticamente desde el último checkpoint\n')

!pip install parse --quiet
!python train.py \
    --model_name TrackNet \
    --epochs {EPOCHS} \
    --save_dir "{EXP_DIR}" \
    --resume_training \
    --verbose

In [ ]:
# ============================================================
# CELDA H — Inferencia sobre tu vídeo de pádel
# Corre si hay modelo entrenado Y vídeo de test en Drive
# Para añadir tu vídeo: sube test_video.mp4 a Drive en:
#   paddelstats_tracknet/test_video.mp4
# ============================================================
import os, subprocess
from pathlib import Path

TRACKNET_FT = f'{EXP_DIR}/TrackNet_best.pt'
INPAINT_PT  = f'{CKPT_DIR}/InpaintNet_best.pt'

# Verificar requisitos
skip = False
if not os.path.exists(TRACKNET_FT):
    print('⚠️  Sin modelo — completa el training (celda G) primero')
    skip = True
if not os.path.exists(TEST_VIDEO):
    print(f'⚠️  Vídeo de test no encontrado en: {TEST_VIDEO}')
    print('   Sube tu vídeo a Drive con ese nombre y vuelve a ejecutar')
    skip = True

if not skip:
    pred_csvs = list(Path(PRED_DIR).glob('*.csv'))
    if pred_csvs:
        print('✅ Predicciones ya existen — saltando inferencia')
        print('   (borra el contenido de predictions/ en Drive para repetirla)')
    else:
        os.chdir(TRACKNET_DIR)
        !pip install parse --quiet

        print('Corriendo inferencia (puede tardar varios minutos)...')
        ret = subprocess.run(
            ['python', 'predict.py',
             '--video_file',      TEST_VIDEO,
             '--tracknet_file',   TRACKNET_FT,
             '--inpaintnet_file', INPAINT_PT,
             '--save_dir',        PRED_DIR,
             '--output_video',
             '--large_video'],
            capture_output=False   # muestra output en tiempo real
        )

        # Verificar que se generó algo
        pred_csvs = list(Path(PRED_DIR).glob('*.csv'))
        if ret.returncode == 0 and pred_csvs:
            print(f'\n✅ Inferencia completada — {len(pred_csvs)} CSV(s) generados en {PRED_DIR}')
        else:
            print(f'\n❌ Inferencia falló o no generó output (returncode={ret.returncode})')
            print(f'   Archivos en {PRED_DIR}: {list(Path(PRED_DIR).iterdir())}')

In [ ]:
# ============================================================
# CELDA I — Resultados
# Muestra métricas si hay predicciones, si no avisa
# ============================================================
import pandas as pd
from pathlib import Path
from google.colab import files

pred_csvs = list(Path(PRED_DIR).glob('*.csv'))
if not pred_csvs:
    print('⚠️  Sin predicciones todavía — ejecuta la celda H primero')
else:
    df = pd.read_csv(pred_csvs[0])
    visible  = df[df['Visibility'] == 1]
    det_rate = len(visible) / len(df) * 100

    print('=' * 50)
    print('RESULTADO FINAL')
    print('=' * 50)
    print(f'Frames procesados : {len(df)}')
    print(f'Pelota detectada  : {len(visible)}')
    print(f'Detection rate    : {det_rate:.1f}%')
    print()
    print('Comparativa:')
    print(f'  YOLOv8 baseline            : ~20%')
    print(f'  TrackNetV3 fine-tuned pádel: {det_rate:.1f}%  ← ESTE')
    print()
    if det_rate > 60:
        print('✅ Listo para integrar en el pipeline de PaddelStats')
    elif det_rate > 30:
        print('⚠️  Mejora real pero insuficiente — prueba EPOCHS=20 en celda G')
    else:
        print('❌ Poca mejora — revisa los logs de entrenamiento (celda G2)')

    pred_videos = list(Path(PRED_DIR).glob('*.mp4'))
    if pred_videos:
        print(f'\nDescargando vídeo anotado...')
        files.download(str(pred_videos[0]))